In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_05_contacts_to_accounts
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_05_contacts_to_accounts
# Spec Ref  : §1.5 Contacts-to-Accounts (3-Phase Relationship Building)
# Purpose   : Build contacts_to_accounts relationship table through 3 phases.
#
# WHY THIS MATTERS (spec §1.5):
#   Without this table you cannot roll up contact-level signals to account level.
#   Account scoring (hot accounts, deal probability) depends entirely on this.
#
# PHASE 1 — CRM-defined (highest confidence):
#   SF contact has AccountId set → direct link, trust Salesforce
#   match_type = 'crm_defined'
#
# PHASE 2 — Domain-based matching (business email domains):
#   contact.domain NOT in free email list → match to existing account by domain
#   If no account exists for that domain → create synthetic account
#   match_type = 'domain_match' or 'domain_created'
#   Phases are MUTUALLY EXCLUSIVE: Phase 2 contacts are LEFT ANTI JOINed
#   against Phase 1 so no contact can appear twice.
#
# PHASE 3 — Free email handling (gmail, yahoo, hotmail etc.):
#   Personal email domain → one synthetic account per email address
#   match_type = 'free_email_match'
#   Excluded from Phase 1 AND Phase 2 via LEFT ANTI JOIN.
#
# Key fix: Uses ROW_NUMBER() not FIRST(ORDER BY) — Spark doesn't support
#   FIRST(col ORDER BY ...) as a standalone aggregate function.
#
# Inputs  : hgi.silver.contacts_all, hgi.silver.accounts_all
# Output  : hgi.silver.contacts_to_accounts
#
# Run after: map_01 (contacts_all), map_02 (accounts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2 ── Widget
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()

print(f"=== Map 05: contacts_to_accounts (3-phase) ===  customer={customer_id}")
print(f"  Contacts : {sv}.contacts_all")
print(f"  Accounts : {sv}.accounts_all")
print(f"  Output   : {sv}.contacts_to_accounts")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"

import sys, os
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
if os.path.abspath(project_root) not in sys.path:
    sys.path.append(os.path.abspath(project_root))

from utils.pipeline_metrics import PipelineMetrics
pm = PipelineMetrics(
    spark          = spark,
    parent_run_id  = parent_run_id,
    job_name       = "02b_map_05_contacts_to_accounts",
    task_key       = "run_job_D_silver_map",
    source_system  = source_system,
    load_type      = load_type,
    customer_id    = customer_id,
    object_name    = object_name,
)
pm.init()

initialize_audit_tables()

In [0]:
# CELL 3 ── Create output table
create_map_table(
    f"{sv}.contacts_to_accounts",
    """
        contact_id  STRING NOT NULL,
        account_id  STRING NOT NULL,
        match_type  STRING NOT NULL,
        tenant      BIGINT
    """,
    cluster_by="contact_id"
)

In [0]:

# CELL 4 ── Prepare free email domain broadcast
free_email_df = spark.createDataFrame(
    [(d,) for d in FREE_EMAIL_DOMAINS],     # FREE_EMAIL_DOMAINS from pipeline_config
    ["domain"]
)
free_email_df.createOrReplaceTempView("free_email_domains_ref")
print(f"  Free email domains loaded: {len(FREE_EMAIL_DOMAINS)}")

In [0]:
# CELL 5 ── Best account per domain (deterministic — latest data_timestamp wins)
# Uses ROW_NUMBER() not FIRST(ORDER BY) — Spark does not support FIRST(col ORDER BY)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW domain_to_best_account AS
    SELECT domain, account_id
    FROM (
        SELECT
            domain,
            id AS account_id,
            ROW_NUMBER() OVER (
                PARTITION BY domain
                ORDER BY data_timestamp DESC NULLS LAST
            ) AS rk
        FROM {sv}.accounts_all
        WHERE domain IS NOT NULL
          AND TRIM(domain) != ''
    )
    WHERE rk = 1
""")

In [0]:

# CELL 6 ── Phase 1: CRM-defined (Salesforce AccountId on Contact)
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.c2a_phase1")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.c2a_phase1
        USING DELTA AS
        SELECT
            c.id                                                    AS contact_id,
            CONCAT('salesforce_Account_Id_', c.a_accountid)        AS account_id,
            'crm_defined'                                           AS match_type,
            c.tenant
        FROM {sv}.contacts c
        WHERE c.a_accountid IS NOT NULL
          AND TRIM(c.a_accountid) != ''
          AND c.id IS NOT NULL
    """)

    n_p1 = cnt(f"{sv}.c2a_phase1")
    print(f"  Phase 1 (crm_defined): {n_p1:,} contacts")
except Exception as e:
    print(f"❌ Phase 1 (crm_defined) failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"Phase 1 failed: {str(e)[:450]}",
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 7 ── Phase 2: Domain-based matching for BUSINESS email domains
# Excludes contacts already linked in Phase 1 (LEFT ANTI JOIN)
# Excludes contacts with free/personal email domains
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.c2a_phase2")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.c2a_phase2
        USING DELTA AS
        SELECT
            c.id        AS contact_id,
            COALESCE(
                da.account_id,
                CONCAT('madkudu_map_domain_', c.domain)   -- synthetic account for new domain
            )           AS account_id,
            CASE
                WHEN da.account_id IS NOT NULL THEN 'domain_match'
                ELSE 'domain_created'
            END         AS match_type,
            c.tenant
        FROM {sv}.contacts c

        -- Exclude Phase 1 contacts — mutually exclusive phases
        LEFT ANTI JOIN {sv}.c2a_phase1 p1
            ON c.id = p1.contact_id

        -- Exclude personal/free email domains (handled in Phase 3)
        LEFT ANTI JOIN free_email_domains_ref fe
            ON c.domain = fe.domain

        -- Match by domain to best existing account
        LEFT JOIN domain_to_best_account da
            ON c.domain = da.domain

        WHERE c.domain IS NOT NULL
          AND TRIM(c.domain) != ''
          AND c.id IS NOT NULL
    """)

    n_p2 = cnt(f"{sv}.c2a_phase2")
    print(f"  Phase 2 (domain match/created): {n_p2:,} contacts")
except Exception as e:
    print(f"❌ Phase 2 (domain match) failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"Phase 2 failed: {str(e)[:450]}",
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 8 ── Phase 3: Free email handling — one synthetic account per email
# Excludes contacts already linked in Phase 1 (LEFT ANTI JOIN)
# Only includes contacts with personal/free email domains
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.c2a_phase3")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.c2a_phase3
        USING DELTA AS
        SELECT
            c.id                                        AS contact_id,
            CONCAT('madkudu_map_email_', c.email)       AS account_id,
            'free_email_match'                          AS match_type,
            c.tenant
        FROM {sv}.contacts c

        -- Exclude Phase 1 contacts — mutually exclusive
        LEFT ANTI JOIN {sv}.c2a_phase1 p1
            ON c.id = p1.contact_id

        -- Only free email domains
        INNER JOIN free_email_domains_ref fe
            ON c.domain = fe.domain

        WHERE c.email IS NOT NULL
          AND TRIM(c.email) != ''
          AND c.id IS NOT NULL
    """)

    n_p3 = cnt(f"{sv}.c2a_phase3")
    print(f"  Phase 3 (free_email_match): {n_p3:,} contacts")
except Exception as e:
    print(f"❌ Phase 3 (free_email) failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"Phase 3 failed: {str(e)[:450]}",
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 9 ── Combine all 3 phases (mutually exclusive → no dedup needed)
try:
    # Safe drop in case target exists as a VIEW
    safe_drop_view(f"{sv}.contacts_to_accounts")

    spark.sql(f"""
        CREATE OR REPLACE TABLE {sv}.contacts_to_accounts
        USING DELTA
        CLUSTER BY (contact_id)
        {DELTA_TBLPROPS_MAP}
        AS
        SELECT * FROM {sv}.c2a_phase1
        UNION ALL
        SELECT * FROM {sv}.c2a_phase2
        UNION ALL
        SELECT * FROM {sv}.c2a_phase3
    """)
except Exception as e:
    print(f"❌ contacts_to_accounts combine failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = f"Combine phases failed: {str(e)[:450]}",
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise

In [0]:
# CELL 10 ── Cleanup phase tables (no longer needed)
spark.sql(f"DROP TABLE IF EXISTS {sv}.c2a_phase1")
spark.sql(f"DROP TABLE IF EXISTS {sv}.c2a_phase2")
spark.sql(f"DROP TABLE IF EXISTS {sv}.c2a_phase3")

In [0]:
# CELL 11 ── Spec DQ gates
try:
    total_contacts  = cnt(f"{sv}.contacts_all")
    total_linked    = cnt(f"{sv}.contacts_to_accounts")
    linkage_pct     = round(100 * total_linked / max(total_contacts, 1), 1)

    orphan_contacts = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.contacts_all c ON c2a.contact_id = c.id
        WHERE c.id IS NULL
    """).collect()[0][0]

    orphan_accounts_crm = spark.sql(f"""
        SELECT COUNT(*) FROM {sv}.contacts_to_accounts c2a
        LEFT JOIN {sv}.accounts_all a ON c2a.account_id = a.id
        WHERE a.id IS NULL AND c2a.match_type = 'crm_defined'
    """).collect()[0][0]

    dupe_contacts = spark.sql(f"""
        SELECT COUNT(*) FROM (
            SELECT contact_id, COUNT(*) AS n
            FROM {sv}.contacts_to_accounts
            GROUP BY contact_id HAVING n > 1
        )
    """).collect()[0][0]

    threshold = DQ_THRESHOLDS["c2a_linkage_pct"]
    print(f"\n  contacts_to_accounts: {total_linked:,} rows")
    print(f"  Linkage: {total_linked:,}/{total_contacts:,} = {linkage_pct}%  (spec ≥{threshold}%)  {'✅ PASS' if linkage_pct >= threshold else '⚠ BELOW THRESHOLD'}")
    print(f"  Orphan contact_ids (crm): {orphan_contacts}  (spec = 0)  {'✅' if orphan_contacts == 0 else '❌'}")
    print(f"  Orphan account_ids (crm_defined): {orphan_accounts_crm}  (spec = 0)  {'✅' if orphan_accounts_crm == 0 else '❌'}")
    print(f"  Duplicate contact_ids: {dupe_contacts}  (phases should be exclusive)  {'✅' if dupe_contacts == 0 else '❌'}")

    print("\n  Match type breakdown:")
    spark.sql(f"""
        SELECT match_type, COUNT(*) AS cnt
        FROM {sv}.contacts_to_accounts
        GROUP BY match_type ORDER BY cnt DESC
    """).show()
except Exception as e:
    print(f"❌ DQ verification failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "DQ_CHECK_FAILED",
        error_type     = type(e).__name__,
        error_reason   = f"DQ check failed: {str(e)[:450]}",
    )
    raise

In [0]:
try:
    total_rows = spark.table(f"{sv}.contacts_to_accounts").count()
    pm.save(status="SUCCESS", rows_processed=total_rows)
except Exception as e:
    print(f"❌ Metrics save failed: {e}")
    log_audit(
        job_name       = "02b_map_05_contacts_to_accounts",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    pm.save(status="FAILED", error_reason=str(e))
    raise